In [ ]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from animal_shelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "SNHU1234"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'my-image.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
#    html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Hr(),
    html.Div(
        
#FIXME Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.

    ),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
#FIXME: Set up the features for your interactive data table to make it user-friendly for your client
#If you completed the Module Six Assignment, you can copy in the code you created here 

                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback(Output('datatable-id','data'),
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
## FIX ME Add code to filter interactive data table with MongoDB queries
#
#        
#        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
#        data=df.to_dict('records')
#       
#       
#        return (data,columns)

# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    ###FIX ME ####
    # add code for chart of your choice (e.g. pie chart) #

    #return [
    #    dcc.Graph(            
    #        figure = px.pie(df, names='breed', title='Preferred Animals')
    #    )    
    #]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):  
    if viewData is None:
        return
    elif index is None:
        return
    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]
        
    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]], children=[
                dl.Tooltip(dff.iloc[row,4]),
                dl.Popup([
                    html.H1("Animal Name"),
                    html.P(dff.iloc[row,9])
                ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

In [2]:
# Project Two Dashboard
# Grazioso Salvare – Tom Sweet

from jupyter_dash import JupyterDash
from dash import html, dcc, dash_table
from dash.dependencies import Input, Output
import dash_leaflet as dl
import plotly.express as px
import pandas as pd

from CRUD_Python_Module import AnimalShelter

# Make Dash work correctly in Codio
JupyterDash.infer_jupyter_proxy_config()

# Instantiate CRUD object
shelter = AnimalShelter()


def get_all_animals():
    """Return full animals collection as a DataFrame."""
    data = shelter.read({})
    df = pd.DataFrame(data)
    if "_id" in df.columns:
        df["_id"] = df["_id"].astype(str)
    return df


full_df = get_all_animals()


def run_rescue_query(rescue_type):
    """
    Run Mongo queries for each rescue type using the CRUD module.
    Returns a DataFrame for the selected rescue type.
    """
    # Base filter for all rescue candidates
    base_filter = {
        "animal_type": "Dog",
        "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
    }

    if rescue_type == "Water Rescue":
        base_filter["breed"] = {
            "$in": [
                "Labrador Retriever Mix",
                "Chesapeake Bay Retriever Mix",
                "Newfoundland Mix"
            ]
        }
        base_filter["sex_upon_outcome"] = "Intact Female"

    elif rescue_type == "Mountain or Wilderness Rescue":
        base_filter["breed"] = {
            "$in": [
                "German Shepherd",
                "Alaskan Malamute",
                "Old English Sheepdog",
                "Siberian Husky",
                "Rottweiler"
            ]
        }
        base_filter["sex_upon_outcome"] = "Intact Male"

    elif rescue_type == "Disaster or Individual Tracking":
        base_filter["breed"] = {
            "$in": [
                "Doberman Pinscher",
                "German Shepherd",
                "Golden Retriever",
                "Bloodhound",
                "Rottweiler"
            ]
        }
        base_filter["sex_upon_outcome"] = {
            "$in": ["Intact Male", "Intact Female"]
        }

    else:
        # "All" or reset, just return every record
        return get_all_animals()

    result = shelter.read(base_filter)
    df = pd.DataFrame(result)
    if "_id" in df.columns:
        df["_id"] = df["_id"].astype(str)
    return df


def build_empty_bar_figure():
    """Return an empty bar chart that will not crash Plotly."""
    fig = px.bar(title="No data to display")
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    return fig


def update_map(view_data, selected_rows):
    """Build the Leaflet map children from the current table view."""
    dff = pd.DataFrame.from_dict(view_data)

    # Base map centered on Austin
    if dff.empty:
        return [
            dl.Map(
                style={"width": "1000px", "height": "500px"},
                center=[30.75, -97.48],
                zoom=10,
                children=[dl.TileLayer(id="base-layer-id")],
            )
        ]

    if not selected_rows:
        row = 0
    else:
        row = selected_rows[0]

    return [
        dl.Map(
            style={"width": "1000px", "height": "500px"},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[
                        dff.iloc[row]["location_lat"],
                        dff.iloc[row]["location_long"],
                    ],
                    children=[
                        dl.Tooltip(dff.iloc[row]["breed"]),
                        dl.Popup(
                            [
                                html.H1("Animal Name"),
                                html.P(dff.iloc[row]["name"]),
                            ]
                        ),
                    ],
                ),
            ],
        )
    ]


# Build Dash app and layout
app = JupyterDash(__name__)

app.layout = html.Div(
    children=[
        html.Div(
            [
                html.H1("Grazioso Salvare Dashboard – Tom Sweet"),
                # Update src if your logo file has a different name or path
                html.Img(
                    id="gs-logo",
                    src="assets/Grazioso_Salvare_Logo.png",
                    style={"height": "60px"},
                ),
            ]
        ),
        html.Hr(),

        # Filter controls on the left
        html.Div(
            [
                html.Label("Select rescue type"),
                dcc.RadioItems(
                    id="rescue-type",
                    options=[
                        {"label": "All", "value": "All"},
                        {"label": "Water Rescue", "value": "Water Rescue"},
                        {
                            "label": "Mountain or Wilderness Rescue",
                            "value": "Mountain or Wilderness Rescue",
                        },
                        {
                            "label": "Disaster or Individual Tracking",
                            "value": "Disaster or Individual Tracking",
                        },
                    ],
                    value="All",
                    labelStyle={"display": "block"},
                ),
            ],
            style={"width": "20%", "float": "left", "padding": "10px"},
        ),

        # Data table on the right
        html.Div(
            [
                html.H3("Austin Animal Center Outcomes data table"),
                dash_table.DataTable(
                    id="datatable-id",
                    columns=[{"name": i, "id": i} for i in full_df.columns],
                    data=full_df.to_dict("records"),
                    filter_action="native",
                    sort_action="native",
                    page_size=10,
                    row_selectable="single",
                    selected_rows=[0],
                    style_table={"overflowX": "scroll"},
                ),
            ],
            style={"width": "75%", "display": "inline-block", "padding": "10px"},
        ),

        # Map and bar chart
        html.Div(
            [
                html.Div(
                    [
                        html.H3("Geolocation map for selected animal"),
                        html.Div(id="map-id"),
                    ],
                    style={
                        "width": "60%",
                        "display": "inline-block",
                        "padding": "10px",
                        "verticalAlign": "top",
                    },
                ),
                html.Div(
                    [
                        html.H3("Rescue candidates by breed"),
                        dcc.Graph(id="breed-graph"),
                    ],
                    style={
                        "width": "35%",
                        "display": "inline-block",
                        "padding": "10px",
                        "verticalAlign": "top",
                    },
                ),
            ],
            style={"clear": "both"},
        ),

        html.Div(
            "Dashboard created by Tom Sweet",
            style={"padding": "10px"},
        ),
    ]
)


# Callback 1: update table when rescue type changes
@app.callback(
    Output("datatable-id", "data"),
    [Input("rescue-type", "value")],
)
def update_table(rescue_type):
    df = run_rescue_query(rescue_type)
    return df.to_dict("records")


# Callback 2: update map when table selection changes
@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "data"),
        Input("datatable-id", "selected_rows"),
    ],
)
def display_map(view_data, selected_rows):
    return update_map(view_data, selected_rows)


# Callback 3: update bar chart from current table view
@app.callback(
    Output("breed-graph", "figure"),
    [Input("datatable-id", "data")],
)
def update_graph(view_data):
    dff = pd.DataFrame.from_dict(view_data)

    if dff.empty or "breed" not in dff.columns:
        return build_empty_bar_figure()

    breed_counts = dff["breed"].value_counts().reset_index()
    breed_counts.columns = ["breed", "count"]

    fig = px.bar(
        breed_counts,
        x="breed",
        y="count",
        title="Number of rescue candidates by breed",
    )
    fig.update_layout(xaxis_title="Breed", yaxis_title="Count")
    return fig


# Run app (Codio will open the correct proxy link in a new tab)
app.run_server()



Dash app running on https://karmadolby-marsvalery-3000.codio.io/proxy/8050/
